#  SIMD programming


Enter your name and student ID.

 * Name:
 * Student ID:


# 1. Introduction
* in this notebook you learn how to take advantage of the SIMD instructions of CPUs
* <font color="blue">SIMD</font> stands for _Single Instruction Multiple Data_
* CPUs offer <font color="blue">SIMD registers</font>, each of which can hold several operands (say 4 floats), as well as <font color="blue">SIMD instructions</font>, which operate on all the values in SIMD registers at once
* in the context of SIMD programming, we use "SIMD" and <font color="blue">"vector"</font> almost interchangeably.  We say "vector registers" or "vector operands" in place of "SIMD registers" or "SIMD operands"
* we say "<font color="blue">vectorize</font> this program" to mean "make this program use vector/SIMD instructions".  We casually say "<font color="blue">simdize</font> this program" as well.
* we call an ordinary single value (e.g., a float) a <font color="blue">"scalar"</font>.  In the context of SIMD programming, scalar is the opposite of vector.
* there are various ways to vectorize programs; roughly from high-level to low-level approaches, they are
  * no SIMD programming
    * use libraries that already use SIMD (we do not discuss this further)
  * "High-level" SIMD programming
	* write ordinary loops and hope the compiler does the job
	* use the simd directives of OpenMP or other language constructs that are designed to be compiled into SIMD instructions; SIMD operands never appear explicitly in the program, and all expressions in your program are still of ordinary scalar types
  * "Low-level" SIMD programming
	* use explicit vector types and variables/expressions of vector types, effectively making "SIMD registers" visible in programs
	* use vector intrinsics, functions that almost directly correspond to SIMD instructions
	* use assembly

# 2. How to work on this notebook
* read text, modify cells if necessary, and execute them
* many cells can be executed without modifying them
* cells that need your work are typically preceded by <font color="green">text colored green explaining what you need to do</font>
* problems with non-trivial work have a section header starting with <font color="green">"Problem X :"</font>

# 3. Jupyter AI
* you can ask Jupyter AI when you need help
* open [pd07_simd.chat](pd07_simd.chat) in this folder
* choose `OpenCode` in Persona Chooser
* you can try several models, but `LiteLLM (taulec)/GPT-5.3 Codex (UTokyo Azure)` is recommended for now
* it is instructed to help you but not to give you answers to exercise problems directly; do not try to circumvent it

# 4. Setting up compilers
- the setup is the same as the one we used for OpenMP


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH

# 5. Setting up the extension to submit jobs
- extension to use `%%sub` magic cell


In [ ]:
%load_ext miyabi

# 6. Necessary or useful compiler options
## 6-1. Flags to enable/inhibit vectorization
* SIMD instructions are used when the optimization level is high enough or when a specific flag is given
  * clang : `-fvectorize`
  * nvc : `-Mvect`
  * gcc : `-ftree-vectorize`
* you may want to inhibit vectorization for comparison
  * clang : `-fno-vectorize`
  * nvc : ??? (`-Mnovect` should be it, but it does not seem to have the desired effect)
  * gcc : `-fno-tree-vectorize`
* Arm has two broad categories of SIMD instruction sets, <font color="blue">Advanced SIMD (NEON)</font> and the <font color="blue">Scalable Vector Extension (SVE)</font>
  * NEON : a traditional SIMD instruction set with 128-bit operands (4 x 32 bits or 2 x 64 bits, which is not very powerful)
  * SVE : a modern SIMD instruction set portable across processors with different SIMD widths (i.e., you can write assembly code that does not assume a particular SIMD width)
	* the SIMD width of Grace is still 128 bits
  * the CPU of Miyabi (Grace) supports both (NEON, SVE and SVE2)
* which SIMD instructions are used depends on compiler flags and the host CPU
  * `-mcpu=neoverse-v2` (`-tp neoverse-v2` for nvc) allows NEON, SVE and SVE2 instructions
  * `-mcpu=native` or `-march=native` has the same effect on the Miyabi login node
  * without any of them, the compiler uses only NEON
  * below, we use `-mcpu=neoverse-v2` for clang and `-tp neoverse-v2` for nvc

## 6-2. Flags to report vectorizability
* in SIMD programming, it is critical to know whether the compiler is able to vectorize the loops you want it to vectorize
* serious compilers have options to report it
  * clang(++): `-Rpass=loop-vectorize` and `-Rpass-missed=loop-vectorize`
  * nvc(++): `-Minfo=vect` and `-Mneginfo=vect`
  * gcc (g++): `-fopt-info-vec-optimized` and `-fopt-info-vec-missed`

## 6-3. `-S` option and `asm volatile` are your friends
* in SIMD programming, it is often necessary to look into the generated assembly code
* a few things you want to know
  1. the `-S` option (of clang++, nvc++ and g++) generates assembly code (_xxx_.cc becomes _xxx_.s)
  1. `asm volatile ("...")` is inline assembly, which simply puts ... into the generated assembly code
  1. its true purpose is to write assembly instructions in C code, but here we use it to put landmarks at the beginning and the end of the loop we are interested in.  note that we put `asm volatile ("# ...")`, which is an assembly comment and has no effect. without it, it quickly becomes difficult to locate the assembly code corresponding to the loop
* see [Grace (Neoverse V2) Assembly Cheat Sheet](https://taura.github.io/parallel-distributed-programming/html/grace_assembly_cheat_sheet.html)


# 7. An easily vectorizable code
* let's take a look at code that is the easiest to vectorize

## 7-1. Non-vectorized code
* let's first look at the scalar (non-vectorized) version, by giving an option to inhibit vectorization


In [ ]:
%%writefile axpb_scalar.cc
void axpb_scalar(float a, float * x, float b, int n) {
  asm volatile("# ========== loop begins ==========");
  for (int i = 0; i < n; i++) {
    x[i] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
# will produce axpb_scalar.s
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fno-vectorize -Rpass=loop-vectorize -Rpass-missed=loop-vectorize axpb_scalar.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -Mnovect -Minfo=vect -Mneginfo=vect axpb_scalar.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fno-tree-vectorize -fopt-info-vec-optimized -fopt-info-vec-missed axpb_scalar.cc

In [ ]:
%%bash
cat -n axpb_scalar.s

* details may differ, but you will find something like this for clang (gcc generates simpler code)
```
    27	.LBB0_4:                                // =>This Inner Loop Header: Depth=1
    28		ldp	s2, s3, [x11, #-4]
    29		add	x8, x8, #2
    30		fmadd	s3, s0, s3, s1
    31		fmadd	s2, s0, s2, s1
    32		stp	s2, s3, [x11, #-4]
    33		add	x11, x11, #8
    34		cmp	x10, x8
    35		b.ne	.LBB0_4
```

* <font color="blue">how do you locate your loop?</font>
  * there are two ways
	1. the following reads an assembly file and visualizes its control flow
	2. the code above has inline assembly statements `asm volatile("# ========== loop begins ==========");` and `asm volatile("# ---------- loop ends ----------");`, which are embedded verbatim in the assembly code, so they are easy to find


In [ ]:
import vis_asm
vis_asm.vis_asm_file("axpb_scalar.s")

* observe that the label `.LBB0_4` and `b.ne .LBB0_4` (branch if not equal) form a loop
* `fmadd` computes `a * x + b`, and importantly, its operands are `s` registers (`s0`, `s1`, ...), scalar registers each holding a single single-precision floating point number
* the loop body contains two `fmadd`s. this is the result of a compiler optimization (loop unrolling) that reduces the number of compare/jump instructions executed
* `ldp` and `stp` load and store a _p_air of registers at a time, which is again a consequence of unrolling, not of vectorization
* seeing `fmadd` on `s` registers, we are pretty confident that the code is NOT vectorized

## 7-2. Vectorized version
* now let's look at the vectorized version


In [ ]:
%%writefile axpb_simd.cc
void axpb_simd(float a, float * x, float b, int n) {
  asm volatile("# ========== loop begins ==========");
  for (int i = 0; i < n; i++) {
    x[i] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
# will produce axpb_simd.s
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -Rpass=loop-vectorize -Rpass-missed=loop-vectorize axpb_simd.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -Minfo=vect -Mneginfo=vect axpb_simd.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopt-info-vec-optimized -fopt-info-vec-missed axpb_simd.cc

In [ ]:
%%bash
cat -n axpb_simd.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("axpb_simd.s")

* again, details may differ, but it will look something like this

```
    37	.LBB0_6:                                // =>This Inner Loop Header: Depth=1
    38		ldp	q3, q4, [x11, #-32]
    39		ldp	q5, q6, [x11]
    40		mov	v7.16b, v2.16b
    41		subs	x12, x12, #16
    42		fmla	v7.4s, v3.4s, v0.s[0]
    43		mov	v3.16b, v2.16b
    44		fmla	v3.4s, v4.4s, v0.s[0]
    45		mov	v4.16b, v2.16b
    46		fmla	v4.4s, v5.4s, v0.s[0]
    47		mov	v5.16b, v2.16b
    48		fmla	v5.4s, v6.4s, v0.s[0]
    49		stp	q7, q3, [x11, #-32]
    50		stp	q4, q5, [x11], #64
    51		b.ne	.LBB0_6
```

* `v` registers (`v7`, `v3`, `v4`, and `v5`) are SIMD registers (`.4s` means _four_ _s_ingle-precision floats), and the `fmla` instructions operate on them.  the same registers are called `q3`, `q4`, ... (128 bits as a whole) in the load/store instructions. operands like `v3.4s`, instead of `s3`, are the hallmark of vectorized code
* the generated code contains many other instructions and looks complex. this is because the trip count may not be a multiple of the SIMD width, in which case there must be remainder iterations executed by scalar instructions before or after the vectorized iterations.  that is,
```
for (i = 0; i < n; i++) {
  S(i);
}
```
becomes something like
```
for (i = 0; i + L <= n; i += L) {
  S(i:i+L);
}
for (; i < n; i++) {
  S(i);
}
```

* just to make our investigation easier, let's eliminate the remainder iterations by forcibly making n a multiple of 16, which is a multiple of the SIMD width (4 floats).  to be sure, this does not preserve the behavior of the original scalar loop.  we nevertheless do so, just to make the generated code simpler and easier to look into.
* `n = (n / 16) * 16` in the code below does the trick


In [ ]:
%%writefile axpy_simd_no_remainder.cc
void axpb_simd_no_remainder(float a, float * x, float b, int n) {
  n = (n / 16) * 16;    /* just so that there are no scalar iterations */
  asm volatile("# ========== loop begins ==========");
  for (int i = 0; i < n; i++) {
    x[i] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -Rpass=loop-vectorize -Rpass-missed=loop-vectorize axpy_simd_no_remainder.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -Minfo=vect -Mneginfo=vect axpy_simd_no_remainder.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopt-info-vec-optimized -fopt-info-vec-missed axpy_simd_no_remainder.cc

In [ ]:
%%bash
cat -n axpy_simd_no_remainder.s

* the visualization looks much simpler thanks to the absence of the remainder loop


In [ ]:
import vis_asm
vis_asm.vis_asm_file("axpy_simd_no_remainder.s")

# 8. When vectorization succeeds or fails
* simple as it may sound, there are many ways in which vectorization fails
* in this section, you will test many code examples that we hope compilers vectorize, and see whether they actually do

## 8-1. Dependencies
* the most typical is a loop that has _dependencies_ between iterations, so that executing them simultaneously changes the behavior of the loop
* compilers fail to vectorize this code


In [ ]:
%%writefile dependency.cc
void dependency(float a, float * x, float b, int n) {
  asm volatile("# ========== loop begins ==========");
  for (int i = 0; i < n - 1; i++) {
    x[i+1] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

* check the compiler message to see whether the code is vectorized
* note: GCC generates an informative message explaining why it failed to vectorize it


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -Rpass=loop-vectorize -Rpass-missed=loop-vectorize dependency.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -Minfo=vect -Mneginfo=vect dependency.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopt-info-vec-optimized -fopt-info-vec-missed dependency.cc

In [ ]:
import vis_asm
vis_asm.vis_asm_file("dependency.s")

* what to look for in the assembly code
  * all floating point instructions are `fmadd` on `s` registers, i.e., the loop is not vectorized
  * the loop is unrolled four times, but the four `fmadd`s form a chain; each uses the result of the previous one (e.g., `fmadd s2, ...` is followed by `fmadd s3, s0, s2, s1`, which reads `s2`), which is exactly the dependency between iterations

* the reason is that vectorizing the loop into
```
for (int i = 0; i < n - 1; i += L) {
  x[i+1:i+1+L] = a * x[i:i+L] + b;
}
```
would change the behavior of the original loop.
* to illustrate, let's say the SIMD width L is 4. the first iteration of the vectorized loop performs
```
x[1:5] = a * x[0:4] + b;
```
whereas the corresponding first four iterations of the original code perform
```
i=0 : x[1] = a * x[0] + b;
i=1 : x[2] = a * x[1] + b;
i=2 : x[3] = a * x[2] + b;
i=3 : x[4] = a * x[3] + b;
```
* observe that the second iteration (i=1) reads x[1], which is _updated_ in the first iteration (i=0). this does not happen in the vectorized version

* therefore _the two versions are not equivalent_
* generally speaking, the conditions under which a loop can be vectorized are similar to those under which a loop can be parallelized. after all, both execute multiple iterations simultaneously
* looking at the original code again,
```
for (int i = 0; i < n - 1; i++) {
  x[i+1] = a * x[i] + b;
}
```
the compiler did not vectorize it because different iterations (say i=0 and i=1) read the same element (x[1]) and at least one of them is a write (i=0 writes to x[1])

## 8-2. Potential (uncertain) dependencies
* a similar but less obvious example is this
* it reads array x and writes the results into another array y (`y[i] = a * x[i] + b`), so it looks safe to vectorize, but what if y overlaps with x (say &y[0] = &x[1])?
* then it would be equivalent to
```
x[i+1] = a * x[i] + b
```
creating exactly the same problem as the previous one.
* the compiler cannot tell at compile time whether x and y overlap, as they are just pointers passed from the caller
* try and observe how the compiler responds
* you will see that clang does vectorize the loop.  look at the generated code before the vectorized loop and you will find that it compares the addresses of x and y; it generates both a vectorized loop and a scalar loop and checks _at runtime_ whether the two arrays overlap, executing the scalar loop if they do (this technique is called _loop versioning_)
* the check costs a few instructions per loop (not per iteration), so it is generally cheap, but the compiler gives up versioning when there are many pointers to check, or when it cannot figure out the range of addresses accessed in the loop at all



In [ ]:
%%writefile uncertain_dependency.cc
void uncertain_dependency(float a, float * x, float b, float * y, int n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
  for (int i = 0; i < n; i++) {
    y[i] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -Rpass=loop-vectorize -Rpass-missed=loop-vectorize uncertain_dependency.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -Minfo=vect -Mneginfo=vect uncertain_dependency.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopt-info-vec-optimized -fopt-info-vec-missed uncertain_dependency.cc

In [ ]:
%%bash
cat -n uncertain_dependency.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("uncertain_dependency.s")

* what to look for in the assembly code
  * there are two loops: one with scalar `fmadd`s on `s` registers and another with `fmla`s on `v` registers (`v2.4s`, ...)
  * before them, you will find instructions like
```
sub   x9, x0, x1      // x9 = x - y (difference of the two addresses)
cmn   x9, #63         // compare x9 with -63
b.lo  .LBB0_4         // jump to the vectorized loop unless y is slightly (1 to 63 bytes) ahead of x
```
  * this is the runtime overlap check of loop versioning; the scalar loop is executed only when y is so close to x that the vectorized loop would read elements of x before they are updated through y

## 8-3. `#pragma omp simd` directive
* the `#pragma omp simd` directive tells the compiler to vectorize the loop without worrying about these dependency issues
* https://www.openmp.org/spec-html/5.1/openmpsu49.html#x74-760002.11.5.1
* you need to give the compiler an option to make it recognize the directive
 * clang(++) : `-fopenmp` (enable OpenMP) or `-fopenmp-simd`
 * nvc(++) : `-mp` (enable OpenMP)
 * gcc (g++) : `-fopenmp` (enable OpenMP) or `-fopenmp-simd`



In [ ]:
%%writefile axpy_omp_simd.cc
void axpb_omp_simd(float a, float * x, float b, float * y, int n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (int i = 0; i < n; i++) {
    y[i] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize axpy_omp_simd.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect axpy_omp_simd.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed axpy_omp_simd.cc

In [ ]:
%%bash
cat -n axpy_omp_simd.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("axpy_omp_simd.s")

* what to look for in the assembly code
  * there is only a vectorized loop; the runtime check and the scalar loop are gone, as `#pragma omp simd` told the compiler that vectorizing is safe
  * `fmla v6.4s, v2.4s, v0.s[0]` computes `v6 += v2 * v0[0]` (`v0.s[0]` means the first `s` element of `v0`, i.e., `a`, multiplied with all four lanes of `v2`)
  * as `fmla` adds the product to its first operand, `b` (held in `v1`) is first copied into it by `mov v6.16b, v1.16b`; this is why each `fmla` is preceded by a `mov`
  * the loop processes 16 elements per iteration (`subs x8, x8, #16`); four vectors of four floats each, which the compiler reports as "interleaved count: 4"

* to maximize the chance of vectorization, we put `#pragma omp simd` in all the examples below
* this does not mean the compiler is then able to vectorize any kind of code, as you will see shortly

## 8-4. Reduction
* just as in parallelization, reduction is a common pattern that superficially introduces dependencies but can actually be vectorized with a suitable execution strategy
* for example,
```
s = 0.0;
for (i = 0; i < n; i++) {
  s += x[i];
}
```
can be executed like
```
sv = {0.0, 0.0, ..., 0.0};
for (i = 0; i < n; i++) {
  sv += x[i:i+L];
}
s = sv[0] + ... + sv[L-1];
```



In [ ]:
%%writefile sum.cc
float sum(float * x, int n) {
  n = (n / 16) * 16;
  float s = 0.0;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd reduction(+:s)
  for (int i = 0; i < n; i++) {
    s += x[i];
  }
  asm volatile("# ---------- loop ends ----------");
  return s;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize sum.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect sum.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed sum.cc

In [ ]:
%%bash
cat -n sum.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("sum.s")

* what to look for in the assembly code
  * the loop accumulates into four vector registers (`v0`, `v1`, `v2`, `v3`) with `fadd v.4s`; i.e., the vector `sv` above, replicated four times to process 16 elements per iteration
  * after the loop, the four vectors are added into one (`fadd v0.4s, ...`), and then the four lanes of `v0` are added into a scalar (`fadda s1, p0, s1, z0.s`, an SVE instruction adding all lanes in order)
  * the accumulators start from `movi v1.4s, #128, lsl #24`, which is -0.0 (sign bit only); -0.0 is the true identity of addition (-0.0 + x = x for any x, including x = -0.0)
  * note that the vectorized loop adds the elements in a different order from the original loop, so the result may differ slightly (floating point addition is not associative); `reduction(+:s)` gives the compiler the permission to do so


## 8-5. If statement within a loop
* does an if statement within a loop hamper vectorization?


In [ ]:
%%writefile branch.cc
void branch(float a, float * x, float b, int n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (int i = 0; i < n; i++) {
    if (x[i] > 0.0) {
      x[i] = a * x[i] + b;
    }
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize branch.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect branch.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed branch.cc

In [ ]:
%%bash
cat -n branch.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("branch.s")

* this one is in fact vectorized, and the report says `vectorization width: vscale x 4`.  `vscale` is the hallmark of SVE, which the compiler chose here for its predicated (masked) instructions; you will find predicate registers (`p0`, `p1`, ...) and `z` registers in the generated code
* what to look for in the assembly code (the core of the loop)
```
ldr     z4, [x10]                  // z4 = x[i:i+4]
fcmgt   p1.s, p0/z, z4.s, #0.0     // p1 = (x[i:i+4] > 0.0), a vector of booleans
fmad    z4.s, p0/m, z3.s, z2.s     // z4 = z4 * a + b, computed for ALL lanes
st1w    { z4.s }, p1, [x10]        // store z4 to x[i:i+4], but only lanes where p1 is true
```
  * `z` registers are SVE vector registers (on Grace, 128 bits; their lower halves are the NEON `v` registers)
  * `p` registers are _predicate registers_, holding one boolean per lane; `p0` here is all true (`ptrue p0.s`)
  * the `if` statement is not a branch anymore; the condition becomes a predicate (`fcmgt` sets `p1`), the then-part is computed for all lanes, and the store is _predicated_ by `p1`, so that elements whose condition is false are left unchanged
  * `p0/z` and `p0/m` specify how lanes whose predicate is false are treated: set to zero (`/z`) or left unchanged (merged, `/m`)
  * `[x10, #1, mul vl]` means "x10 + 1 x (the vector length)"; SVE code does not hard-code the vector length; `rdvl` (read the vector length), `decb` and `incb` (decrement/increment by the number of bytes in a vector) also serve this purpose

## 8-6. A loop within a loop (loop-invariant trip count)
* a loop containing another loop whose trip count is a variable
* the trip count is the same across all iterations of the outer loop, so the entire iteration space is a rectangle, and it should be easy to recognize it as such



In [ ]:
%%writefile loop_i.cc
/* inner loop, with a loop-invariant trip count */
void loop_i(float a, float * x, float b, int n, int m) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (int i = 0; i < n; i++) {
    for (int k = 0; k < m; k++) {
      x[i] = a * x[i] + b;
    }
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize loop_i.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect loop_i.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed loop_i.cc

In [ ]:
%%bash
cat -n loop_i.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("loop_i.s")

* what to look for in the assembly code
  * the compiler reports `loop not vectorized` and warns that it could not perform the transformation requested by `#pragma omp simd`
  * the innermost loop consists of scalar `fmadd`s (unrolled eight times), all on the same register (`fmadd s2, s0, s2, s1`)
  * the compiler (clang, at least) vectorizes only innermost loops; vectorizing this code would require vectorizing the outer (i) loop, whose iterations are independent, while each iteration runs the inner (k) loop, which carries a dependency through x[i]
  
* _how to vectorize this loop?_  
  * vectorizing this loop is conceptually as straightforward as below
```
  for (int i = 0; i < n; i += L) {
    for (int k = 0; k < m; k++) {
      x[i:i+L] = a * x[i:i+L] + b;  // (*)
    }
  }
```
  * the issue now is how to use SIMD instruction for the statement (\*)
  * you can do so in two ways
    1. use vector type explained below
    1. express (\*) by a loop and hope the compiler now vectorizes it
```
  for (int i = 0; i < n; i += L) {
    for (int k = 0; k < m; k++) {
	  for (int ii = i; ii < i + L; ii++) {
        x[ii] = a * x[ii] + b;
      }
    }
  }
```
  * now the loop becomes the innermost loop, which the compiler readily vectorizes
  * let's confirm it (with L = 4, the SIMD width)


In [ ]:
%%writefile loop_i_vec.cc
/* loop_i (VER 11) vectorized by hand; the innermost loop now runs
   over L consecutive elements, which the compiler can vectorize */
enum { L = 4 };          /* the SIMD width (4 floats) */
void loop_i_vec(float a, float * x, float b, int n, int m) {
  n = (n / L) * L;
  asm volatile("# ========== loop begins ==========");
  for (int i = 0; i < n; i += L) {
    for (int k = 0; k < m; k++) {
#pragma omp simd
      for (int ii = i; ii < i + L; ii++) {
        x[ii] = a * x[ii] + b;
      }
    }
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize loop_i_vec.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect loop_i_vec.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed loop_i_vec.cc

In [ ]:
%%bash
cat -n loop_i_vec.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("loop_i_vec.s")

* what to look for in the assembly code
  * the compiler may still report `loop not vectorized` (for the k loop), but the generated code shows the innermost loop is vectorized
  * the k loop looks like this
```
ldr     q2, [x11]                    // load x[i:i+4] before the k loop
    ...
mov     v3.16b, v1.16b               // (the k loop, unrolled)
fmla    v3.4s, v2.4s, v0.s[0]        // x[i:i+4] = a * x[i:i+4] + b
mov     v2.16b, v1.16b
fmla    v2.4s, v3.4s, v0.s[0]        // x[i:i+4] = a * x[i:i+4] + b
    ...
str     q2, [x11]                    // store x[i:i+4] after the k loop
```
  * note that x[i:i+4] stays in a register throughout the k loop; it is loaded before and stored after the loop
  * each `fmla` depends on the previous one, so this loop is bound by the latency of `fmla`; a larger L (e.g., 16) makes the k loop update several independent vectors and increases the throughput (we will learn more about this in the next notebook)
  



## 8-7. A loop within a loop (variable trip counts)
* the same as above, but the trip count of the inner loop differs from one iteration of the outer loop to another
* therefore the entire iteration space is not a rectangle



In [ ]:
%%writefile loop_v.cc
/* inner loop, with a variable trip count */
void loop_v(float a, float * x, float b, int n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (int i = 0; i < n; i++) {
    for (int k = 0; k < i; k++) {
      x[i] = a * x[i] + b;
    }
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize loop_v.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect loop_v.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed loop_v.cc

In [ ]:
%%bash
cat -n loop_v.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("loop_v.s")

* what to look for in the assembly code
  * the result is the same as the previous example; not vectorized, and the inner loop consists of scalar `fmadd`s
  * vectorizing the outer loop is even harder here, as different lanes (different i's) would need to execute the inner loop a different number of times

## 8-8. A function call within a loop
* a function call within a loop
* if the body of the function is not visible to the compiler compiling the loop (i.e., it is not in the same compilation unit as the loop), there is no vectorized version of the function
* therefore it is highly unlikely that this code gets meaningfully vectorized


In [ ]:
%%writefile funcall.cc
float f(float a, float x, float b);
void funcall(float a, float * x, float b, int n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (int i = 0; i < n; i++) {
    x[i] = f(a, x[i], b);
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize funcall.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect funcall.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed funcall.cc

In [ ]:
%%bash
cat -n funcall.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("funcall.s")

* what to look for in the assembly code
  * the loop calls the function `f` (`bl _Z1ffff`; `_Z1ffff` is the C++ name of `f(float, float, float)`) once for each element, with scalar arguments in `s0`, `s1` and `s2`
  * the loop is not vectorized

## 8-9. A function call within a loop with `#pragma omp declare simd`
* OpenMP has a directive `#pragma omp declare simd`, which declares that a vectorized version of a function exists
* https://www.openmp.org/spec-html/5.1/openmpsu49.html#x74-780002.11.5.3
* calling such a function within a loop should not hamper vectorization, even if the body of the function is not in the same compilation unit as the loop
* however, clang++ does not seem to be able to vectorize this code (g++ does)


In [ ]:
%%writefile funcall_decl_simd.cc
#pragma omp declare simd uniform(a, b) notinbranch
float f(float a, float x, float b);

void funcall_decl_simd(float a, float * x, float b, int n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (int i = 0; i < n; i++) {
    x[i] = f(a, x[i], b);
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize funcall_decl_simd.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect funcall_decl_simd.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed funcall_decl_simd.cc

In [ ]:
%%bash
cat -n funcall_decl_simd.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("funcall_decl_simd.s")

* what to look for in the assembly code
  * clang generates exactly the same code as without `declare simd`; the loop calls the scalar version of `f` (`bl _Z1ffff`) for each element
  * g++ instead calls a vectorized version of `f`, named `_ZGVnN4uvu__Z1ffff`, which processes four elements per call (`N4`: four lanes; `uvu`: the first and the third arguments are uniform and the second is a vector)

## 8-10. A function defined with `#pragma omp declare simd`
* what happens to a function defined with `#pragma omp declare simd`?
* the idea is that the compiler generates a vectorized version of the function, whose arguments are vectors unless specified as `uniform` (the same across all elements)
* again, clang++ does not seem capable of handling this pragma (g++ does)


In [ ]:
%%writefile fundef_decl_simd.cc
#pragma omp declare simd uniform(a, b) notinbranch
float fundef_decl_simd(float a, float x, float b) {
  return a * x + b;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize fundef_decl_simd.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect fundef_decl_simd.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed fundef_decl_simd.cc

In [ ]:
%%bash
cat -n fundef_decl_simd.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("fundef_decl_simd.s")

* what to look for in the assembly code
  * clang generates only the ordinary scalar function `_Z16fundef_decl_simdfff`
  * g++ additionally generates vectorized versions `_ZGVnN2uvu__Z16fundef_decl_simdfff` and `_ZGVnN4uvu__Z16fundef_decl_simdfff` (two and four lanes), which loops like `funcall_decl_simd` above can call

## 8-11. A strided array access (load)
* an ordinary vector load/store instruction (`ldr` or `str` of a `q` register in NEON) takes a pointer (address) and reads or writes consecutive addresses
* for example, the following load instruction
```
ldr q0, [x0]
```
reads the 16 consecutive bytes starting from the address in the `x0` register and puts the result into the `q0` register
* with this instruction alone, it is difficult to vectorize a loop whose consecutive iterations do not access consecutive addresses
* the simplest example is a strided access like this

* there are two possible strategies
  * access arrays using ordinary vector load instructions and shuffle data around on registers
  * access arrays using `gather` instructions



In [ ]:
%%writefile stride_load.cc
void stride_load(float a, float * x, float b, float * y, int n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (int i = 0; i < n; i++) {
    y[i] = a * x[i * 2] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize stride_load.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect stride_load.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed stride_load.cc

In [ ]:
%%bash
cat -n stride_load.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("stride_load.s")

* what to look for in the assembly code
  * the loads are `ld2 { v3.4s, v4.4s }, [x13]`, which loads eight consecutive floats (x[2i], x[2i+1], ..., x[2i+7]) and _de-interleaves_ them; the elements at even positions go to `v3` and those at odd positions go to `v4`
  * only `v3` (x[2i], x[2i+2], x[2i+4], x[2i+6]) is used by `fmla`; `v4` is simply discarded
  * this is the first strategy above (ordinary loads + shuffling), where the shuffling is built into the load instruction; there are `ld3` and `ld4` instructions as well, for strides 3 and 4

## 8-12. A strided array access (store)

In [ ]:
%%writefile stride_store.cc
void stride_store(float a, float * x, float b, float * y, int n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (int i = 0; i < n; i++) {
    y[i * 2] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize stride_store.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect stride_store.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed stride_store.cc

In [ ]:
%%bash
cat -n stride_store.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("stride_store.s")

* what to look for in the assembly code
  * the computation is vectorized (`fmla v5.4s, v4.4s, v0.s[0]`), but the results are stored one element at a time; `str s5, ...` stores lane 0 and `st1 { v5.s }[1], [x9]` stores lane 1, etc.
  * the addresses of the four elements are computed in vector registers (`index z2.d, #0, #1` builds {0, 1}, and `shl v6.2d, v3.2d, #3` multiplies them by 8) and moved to general purpose registers (`mov x11, v6.d[1]`, `fmov x10, d4`) to be used by the stores
  * the counterpart of `ld2`, `st2`, cannot be used here, as it would also write y[2i+1], y[2i+3], ..., which the loop must not modify

## 8-13. An array of structures
* a less obvious but common pattern is accessing an array of structures
* even if consecutive iterations access consecutive array elements, the actual addresses are not consecutive


In [ ]:
%%writefile struct_load.cc
typedef struct {
  float x;
  float y;
} point_t;
void struct_load(float a, point_t * p, float b, point_t * q, int n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (int i = 0; i < n; i++) {
    q[i].x = a * p[i].x + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize struct_load.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect struct_load.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed struct_load.cc

In [ ]:
%%bash
cat -n struct_load.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("struct_load.s")

* what to look for in the assembly code
  * loading `p[i].x` is done by `ld2 { v3.4s, v4.4s }, [x11], #32`, which loads four structures (eight floats) and de-interleaves them into the x's (`v3`) and the y's (`v4`), just like the strided load above
  * storing `q[i].x` is done one element at a time (`stur s5, ...` and `st1 { v5.s }[k], ...`), just like the strided store above, as `q[i].y` must not be modified
  * the scalar code after the loop handles the last few elements (the compiler is not sure the vectorized loop can read the y of the last structure safely)

## 8-14. An irregular array access
* irregular array accesses, in which consecutive iterations access elements with non-constant strides



In [ ]:
%%writefile non_affine_idx.cc
/* assume n is a power of two, so that i * i % n == (i * i) & (n - 1)
   (% would make the loop dominated by slow vector integer divisions).
   i * i is computed in unsigned, as it may overflow */
void non_affine_idx(float a, float * x, float b, float * y, int n) {
  n = (n / 16) * 16;
  unsigned mask = n - 1;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (int i = 0; i < n; i++) {
    y[i] = a * x[((unsigned)i * i) & mask] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize non_affine_idx.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect non_affine_idx.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed non_affine_idx.cc

In [ ]:
%%bash
cat -n non_affine_idx.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("non_affine_idx.s")

* what to look for in the assembly code (the core of the loop)
```
mul     z7.s, z4.s, z4.s                     // i * i for four values of i
add     z4.s, z4.s, z5.s                     // advance i by 4
and     z7.d, z7.d, z2.d                     // (i * i) & mask
ld1w    { z7.s }, p0/z, [x0, z7.s, uxtw #2]  // gather x[(i * i) & mask]
fmad    z7.s, p0/m, z6.s, z3.s               // * a + b
st1w    { z7.s }, p0, [x1, x11, lsl #2]      // store to y[i:i+4]
```
  * the indices are computed in a vector register (`z4` holds four consecutive values of i, and `mul` and `and` compute the indices)
  * `ld1w { z7.s }, p0/z, [x0, z7.s, uxtw #2]` is a _gather_ instruction (SVE); it loads a 32-bit element from address `x0 + z7[j] * 4` for each lane j, i.e., x[z7[0]], x[z7[1]], x[z7[2]], x[z7[3]] (`uxtw` means each index is zero-extended to 64 bits when the address is computed).  this is the second strategy (gather) mentioned above
  * NEON has no gather instruction; this is another reason the compiler chose SVE here


## 8-15. An indirect array access
* the same as above, but the indices come from another (index) array, a frequently occurring pattern in sparse matrix and graph applications


In [ ]:
%%writefile indirect_idx.cc
void indirect_idx(float a, float * x, int * idx, float b, float * y, int n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (int i = 0; i < n; i++) {
    y[i] = a * x[idx[i]] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize indirect_idx.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 -mp -Minfo=vect -Mneginfo=vect indirect_idx.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed indirect_idx.cc

In [ ]:
%%bash
cat -n indirect_idx.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("indirect_idx.s")

* what to look for in the assembly code (the core of the loop)
```
ld1w    { z4.s }, p0/z, [x1, x10, lsl #2]    // z4 = idx[i:i+4]
ld1w    { z4.s }, p0/z, [x0, z4.s, sxtw #2]  // gather x[idx[i]], ..., x[idx[i+3]]
fmad    z4.s, p0/m, z3.s, z2.s               // * a + b
st1w    { z4.s }, p0, [x2, x10, lsl #2]      // store to y[i:i+4]
```
  * the indices are first loaded from `idx` by an ordinary (contiguous) vector load
  * the rest is similar to the previous example (`sxtw` instead of `uxtw`, as the indices are signed `int`s)



# 9. Vector types
## 9-1. When you need vector types
* in an ideal world, you just write loops and the compiler vectorizes them, but it often fails, and we are rarely certain that it succeeds
* in such cases we resort to "low-level SIMD programming", which explicitly manipulates and operates on "vectors of values" and is almost certainly compiled into SIMD instructions
* the gcc, clang, and nvc compilers allow you to define types representing vectors of values and to operate on them
* see [GCC manual](https://gcc.gnu.org/onlinedocs/gcc/Vector-Extensions.html) and [Clang manual](https://clang.llvm.org/docs/LanguageExtensions.html#vectors-and-extended-vectors) for reference

* the following code defines types representing vectors of four, eight, and sixteen floats (`float_4`, `float_8`, and `float_16`, respectively) and functions that take two such values
* these types support the familiar syntax for many arithmetic expressions (+, -, *, /, etc.)


In [ ]:
%%writefile vector_ext.cc
typedef float float_4 __attribute__((vector_size(16),__may_alias__,aligned(sizeof(float))));
typedef float float_8 __attribute__((vector_size(32),__may_alias__,aligned(sizeof(float))));
typedef float float_16 __attribute__((vector_size(64),__may_alias__,aligned(sizeof(float))));

float_4 dist_4(float_4 x, float_4 y) {
  return x * x + y * y;
}
float_8 dist_8(float_8 x, float_8 y) {
  return x * x + y * y;
}
float_16 dist_16(float_16 x, float_16 y) {
  return x * x + y * y;
}


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
# will produce vector_ext.s
clang++ -S -Wall -O3 -mcpu=neoverse-v2 vector_ext.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 vector_ext.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 vector_ext.cc

In [ ]:
%%bash
cat -n vector_ext.s

* visualize the control flow graph of the assembly code


In [ ]:
import vis_asm
vis_asm.vis_asm_file("vector_ext.s")

* conveniently, the size of a vector type (16, 32, 64 bytes) does not have to match the native size of the vector registers of the CPU (16 bytes)
* the compiler is able to handle vector types larger than the native size by emitting as many instructions as necessary

* you can mix vectors and scalars in arithmetic expressions


In [ ]:
%%writefile scalar_vector.cc
typedef float float_16 __attribute__((vector_size(64),__may_alias__,aligned(sizeof(float))));

float_16 axpb_16(float a, float_16 x) {
  return a * x + 3.0;
}


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
# will produce scalar_vector.s
clang++ -S -Wall -O3 -mcpu=neoverse-v2 scalar_vector.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 scalar_vector.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 scalar_vector.cc

In [ ]:
%%bash
cat -n scalar_vector.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("scalar_vector.s")

## 9-2. A note about `aligned(sizeof(float))`
* if you are curious, `aligned(sizeof(float))` sets the minimum alignment of the type to the size of a float (4), which is convenient as `floatv` will be used to work on arrays of floats
* without it, the alignment becomes the size of `floatv` (16)
* AArch64 `ldr`/`str` of a `q` register do not require the address to be 16-byte aligned, so unlike x86, there is no pair of aligned/unaligned instructions to choose between
* the alignment still matters to the type system, as converting a `float *` into a `floatv *` is well defined only when the alignment requirement of `floatv` is met

# 10. Programming with vector types
* once you have defined a vector type, there is not much to learn other than how to build a vector from scalars and vice versa

## 10-1.  Building a vector value from scalars
### 10-1-1. From an array of scalars
* if you access an array of scalars (i.e., `float *`) through a pointer to a vector (e.g., `floatv *`), you get a vector of consecutive elements starting from the given address


In [ ]:
%%writefile loadv.cc
typedef float floatv __attribute__((vector_size(simd_width),__may_alias__,aligned(sizeof(float))));

floatv loadv(float * a) {
  /* a vector {a[0],a[1],...,a[L-1]} */
  return *((floatv *)a);
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
# will produce loadv.s
clang++ -S -Wall -O3 -mcpu=neoverse-v2 loadv.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 loadv.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 loadv.cc

In [ ]:
%%bash
cat -n loadv.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("loadv.s")

### 10-1-2. Building a vector of arbitrary values
* you can build a vector of arbitrary values just by making an array of these values and applying the method above


In [ ]:
%%writefile make_vector.cc
typedef float floatv __attribute__((vector_size(simd_width),__may_alias__,aligned(sizeof(float))));

floatv make_vector(float a0, float a1, float a2, float a3) {
  float a[4] = { a0, a1, a2, a3 };
  return *((floatv *)a);
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
# will produce make_vector.s
clang++ -S -Wall -O3 -mcpu=neoverse-v2 make_vector.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 make_vector.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 make_vector.cc

In [ ]:
%%bash
cat -n make_vector.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("make_vector.s")

* you will later learn that there is an intrinsic function exactly for this

## 10-2. Extracting a scalar value from a vector
### 10-2-1. To an array of scalars
* you can store a vector into an array of scalars


In [ ]:
%%writefile storev.cc
typedef float floatv __attribute__((vector_size(simd_width),__may_alias__,aligned(sizeof(float))));

void storev(float * a, floatv v) {
  *((floatv *)a) = v;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
# will produce storev.s
clang++ -S -Wall -O3 -mcpu=neoverse-v2 storev.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 storev.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 storev.cc

In [ ]:
%%bash
cat -n storev.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("storev.s")

### 10-2-2. Index notation
* an element of a vector can be extracted with the index notation for arrays, but internally, the compiler typically stores the vector into a temporary array and then extracts the element you want, which is not terribly efficient


In [ ]:
%%writefile get_i.cc
typedef float floatv __attribute__((vector_size(simd_width),__may_alias__,aligned(sizeof(float))));

float get_i(floatv v, int i) {
  return v[i];
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
# will produce get_i.s
clang++ -S -Wall -O3 -mcpu=neoverse-v2 get_i.cc
#nvc++ -S -Wall -O3 -tp neoverse-v2 get_i.cc
#g++ -S -Wall -O3 -mcpu=neoverse-v2 get_i.cc

In [ ]:
%%bash
cat -n get_i.s

In [ ]:
import vis_asm
vis_asm.vis_asm_file("get_i.s")

# 11. Vector intrinsics
## 11-1. When you have to use vector intrinsics
* vector intrinsics are functions that directly manipulate vector types and are compiled almost directly into the corresponding SIMD instructions
* remember that with the vector extension, you can have vectors of values and write arithmetic expressions involving them; you do not have to use vector intrinsics where it applies
* however, there are many operations that have no convenient C expressions. e.g.,
  * comparisons between two vectors; given `A` and `B` of type `floatv`, one wishes `A < B` were a vector of booleans or ints {A[0]<B[0], A[1]<B[1], ..., A[L-1]<B[L-1]}, but it isn't
  * math functions such as log, exp, etc.; given `V` of type `floatv`, one wishes `sqrt(V)` returned a value of `floatv` by applying sqrt to each element of `V`, but it doesn't (a SIMD sqrt instruction does exist on Arm CPUs)
  * array indexing with a vector of integers; given `I` of type `intv` (a vector of ints, defined like `floatv` above) and `a` of type `float*`, one wishes `a[I]` were a value of `floatv` {a[I[0]], a[I[1]], ..., a[I[L-1]]}, but it isn't
* in all these circumstances, if high-level programming does not work, you need vector intrinsics

## 11-2. Reference
* https://developer.arm.com/architectures/instruction-sets/intrinsics/
* you will be overwhelmed by the sheer number of functions; you never have to remember all of them
* just remember that
  * most functions correspond to a SIMD instruction, so what you reasonably hope exists probably exists
    * e.g., want to do a SIMD sqrt? just search for sqrt and look for what you need
  * the page covers both NEON and SVE/SVE2, so check which one a function belongs to before using it

* with that said, here is an overview

## 11-3. The header file
* include 
```
#include <arm_neon.h>
#include <arm_sve.h>
```
for using NEON and SVE

## 11-4. Types and their naming conventions
* type names are of the form `<element type><bits per element>x<number of elements>_t`
  * `float64x2_t` : vector of 2 x 64-bit (`double`) floating point numbers (128 bits)
  * `float32x4_t` : vector of 4 x 32-bit (`float`) floating point numbers (128 bits)
  * `float32x2_t` : vector of 2 x 32-bit (`float`) floating point numbers (64 bits)
  * `int64x2_t` : vector of 2 x 64-bit (`long`) integers (128 bits)
  * `int32x4_t` : vector of 4 x 32-bit (`int`) integers (128 bits)
  * `int16x8_t` : vector of 8 x 16-bit (`short`) integers (128 bits)
  * `int8x16_t` : vector of 16 x 8-bit (`char`) integers (128 bits)
  * there are unsigned versions (`uint`), too

* there are _sizeless_ types used for SVE instructions, which begin with `sv` and have no `xN` part
  * `svfloat64_t` : vector of 64-bit (`double`) floating point numbers (size unknown)
  * `svfloat32_t` : vector of 32-bit (`float`) floating point numbers (size unknown)
  * `svint64_t` : vector of 64-bit (`long`) integers (size unknown)
  * `svint32_t` : vector of 32-bit (`int`) integers (size unknown)
  * `svint16_t` : vector of 16-bit (`short`) integers (size unknown)
  * `svint8_t` : vector of 8-bit (`char`) integers (size unknown)
  * `svbool_t` : vector of bools (size unknown)
* these sizeless types serve as operand types for SVE instructions, and their sizes are literally _unknown_ at compile time
  * you cannot apply the `sizeof` operator to them
  * you cannot have arrays of them, structures containing them, or global variables of them
  * you cannot apply the index operator to pointers to them (i.e., p[i])
  * more on this later

## 11-5. Functions and their naming conventions
* function names begin with `v`, followed by the operation, followed by `q` when it operates on a whole 128-bit register, and end with the element type. e.g.,
  * `float32x4_t vfmaq_f32 (float32x4_t a, float32x4_t b, float32x4_t c)` takes three 128 bit vectors of floats and performs fma (a + b * c)
  * `float64x2_t vmaxq_f64 (float64x2_t a, float64x2_t b)` takes the element-wise maximum of two vectors of doubles

## 11-6. Dup, load and store
* `vdupq_n_f32` takes a float value and returns a 4-element vector of it. e.g., `vdupq_n_f32(3)` returns `{3,3,3,3}`
* `vld1q_f32` and `vst1q_f32` load and store a vector from/to a `float *`
* `vgetq_lane_f32` and `vsetq_lane_f32` read and write a single lane
* similar functions exist for the other element types (e.g., `vdupq_n_f64` for doubles)

* note that you do not need any of them as long as you stay within the vector extension; `*((floatv *)a)` and `v[i]` do the same job

## 11-7. A brief categorization
In parentheses is the category to check on the Arm Intrinsics Reference page

* arithmetic, which you can do with the vector extension alone, without intrinsics (Arithmetic)
* math functions (Math)
* comparison (Compare)
* shuffling values on vectors (Move, Permutation)
* memory access (Load, Store)
